In [0]:
from src.transforms.gold import build_dim_date, latest_actor_state, event_type_dim

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("start_date", "2026-07-07")
dbutils.widgets.text("end_date", "2026-07-08")
CATALOG = dbutils.widgets.get("catalog")
START = dbutils.widgets.get("start_date")
END = dbutils.widgets.get("end_date")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

events = spark.table(f"{CATALOG}.silver.events")

build_dim_date(spark, "2026-05-01", "2026-12-31").write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_date")

latest_actor_state(events).write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_actor")
event_type_dim(events).write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_event_type")
print("dim built")


In [0]:
spark.sql(f"SELECT count(*) AS days FROM {CATALOG}.gold.dim_date").show()

spark.sql(f"SELECT count(*) AS actors, sum(cast(is_bot AS int)) AS bots FROM {CATALOG}.gold.dim_actor").show()
spark.sql(f"SELECT * FROM {CATALOG}.gold.dim_event_type ORDER BY event_type").show(30)
spark.sql(f"""
          SELECT date_key, date, day_name, is_weekend
          FROM {CATALOG}.gold.dim_date WHERE date = '2026-06-01'
          """).show()

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.gold.dim_repo (
              repo_sk BIGINT GENERATED ALWAYS AS IDENTITY,
              repo_id BIGINT NOT NULL,
              repo_name STRING NOT NULL,
              owner STRING,
              language STRING,
              effective_from TIMESTAMP,
              effective_to TIMESTAMP,
              is_current BOOLEAN NOT NULL
          )""")

In [0]:
from src.transforms.gold import latest_repo_state
from src.writers.gold_writer import scd2_upsert_dim_repo
events = spark.table(f"{CATALOG}.silver.events")
scd2_upsert_dim_repo(latest_repo_state(events), CATALOG)

spark.sql(f"""
          SELECT count(*) total, sum(cast(is_current AS int)) current_rows
          FROM {CATALOG}.gold.dim_repo
          """).show()
          

In [0]:
spark.sql(f"SELECT count(*) FROM {CATALOG}.gold.dim_repo WHERE language IS NOT NULL").show()

In [0]:
# --- (SCD2 rename drill / cleanup): disabled for scheduled job runs ---
# from pyspark.sql import functions as F
# from src.writers.gold_writer import scd2_upsert_dim_repo
# 
# def _drill(name):
#     return (
#         spark.createDataFrame(
#     [(999999901, name, "drill-org", None)],
#     "repo_id bigint, repo_name string, owner string, language string",
#     )
#     .withColumn("observed_at", F.current_timestamp())
#     .withColumn("first_seen_ts", F.current_timestamp())
#     .select("repo_id", "repo_name", "owner", "observed_at", "first_seen_ts", "language")
#     )
# 
# scd2_upsert_dim_repo(_drill("drill-org/original-name"), CATALOG)
# scd2_upsert_dim_repo(_drill("drill-org/renamed"), CATALOG)
# 
# spark.sql(f"""
#           SELECT repo_sk, repo_name, effective_from, effective_to, is_current
#           FROM {CATALOG}.gold.dim_repo WHERE repo_id = 999999901 ORDER BY effective_from
#           """).show(truncate = False)

In [0]:
# ---(SCD2 rename drill / cleanup): disabled for scheduled job runs ---
# spark.sql(f"DELETE FROM {CATALOG}.gold.dim_repo WHERE repo_id = 999999901")

In [0]:
spark.sql(f"""
          SELECT count(*) AS pr_events,
          count(get_json_object(payload, '$.pull_request.base.repo.language' )) AS with_lang
          FROM {CATALOG}.silver.events
          WHERE event_type = 'PullRequestEvent'
          """).show()
        
spark.sql(f"""
          SELECT substring(payload, 1, 600) AS payload_head
          FROM {CATALOG}.silver.events
          WHERE event_type = 'PullRequestEvent' LIMIT 2
          """).show(truncate=False)

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.gold.fact_events (
              event_id STRING NOT NULL, date_key INT, event_type STRING, actor_id BIGINT,
              repo_id BIGINT, event_hour INT, action STRING, event_date DATE
          )PARTITIONED BY (event_date)""")

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.gold.fact_repo_daily_metrics (
              repo_id BIGINT NOT NULL, event_date DATE, date_key INT, events_total BIGINT,
              stars BIGINT, forks BIGINT, pushes BIGINT, commits_pushed BIGINT,
              prs_opened BIGINT, prs_merged BIGINT, issues_opened BIGINT, releases BIGINT,
              unique_actors BIGINT, unique_human_actors BIGINT
          )PARTITIONED BY(event_date) """)

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.gold.fact_language_trends (
              event_date DATE, date_key INT, language STRING, active_repos BIGINT,
              events BIGINT, share_of_events DOUBLE
          )PARTITIONED BY (event_date)""")
          

In [0]:
from src.transforms.gold import build_fact_events, daily_repo_metrics, language_trends
from src.writers.gold_writer import overwrite_date_range


events = spark.table(f"{CATALOG}.silver.events").filter(f"event_date BETWEEN '{START}' AND '{END}'")

overwrite_date_range(build_fact_events(events), f"{CATALOG}.gold.fact_events", START, END)
overwrite_date_range(daily_repo_metrics(events), f"{CATALOG}.gold.fact_repo_daily_metrics", START, END)

dim_repo = spark.table(f"{CATALOG}.gold.dim_repo")
overwrite_date_range(language_trends(events, dim_repo), f"{CATALOG}.gold.fact_language_trends", START, END)
print("fact built")

In [0]:
for t in ["fact_events", "fact_repo_daily_metrics", "fact_language_trends"]:
    print(f"{t:26s}", spark.table(f"{CATALOG}.gold.{t}").count())

spark.sql(f"""
          SELECT repo_id, events_total, stars, forks, pushes, prs_opened, unique_human_actors
          FROM {CATALOG}.gold.fact_repo_daily_metrics ORDER BY events_total DESC LIMIT 10
          """).show()
          

In [0]:
spark.sql(f"""
          SELECT sum(events_total) events, sum(stars) stars, sum(forks) forks,
          sum(pushes) pushes, sum(prs_opened) prs_opened, sum(prs_merged) prs_merged, sum(issues_opened) issues_opened, sum(releases) releases FROM {CATALOG}.gold.fact_repo_daily_metrics""").show()

In [0]:
spark.sql(f"""
          SELECT repo_id, stars, forks, prs_opened, events_total
          FROM {CATALOG}.gold.fact_repo_daily_metrics
          WHERE stars > 0 ORDER BY stars DESC LIMIT 10""").show()
          

In [0]:
spark.sql(f"""
          SELECT event_type, action, count(*) c 
          FROM {CATALOG}.silver.events WHERE event_type IN
          ('IssuesEvent', 'PullRequestEvent') GROUP BY event_type, action ORDER BY event_type, c DESC""").show()